# Census Data Imports (Work in Progress)

By Kenneth Burchfiel

Released under the MIT License

The US Census is a fantastic source of free demographic data. As this notebook will demonstrate, Python allows us to easily access large amounts of this data at once.

## Deciding where to move to start a family

Let's say that some NVCU graduating seniors are interested in settling down and raising a family a few years after they graduate. Because they'd prefer to live in a growing region rather than a declining one, they want to know which areas have seen the highest growth rates in recent years. They'd like to see this data both for all residents within each county *and* those aged 25-29.

In order to answer these questions, we'll use the Census API to retrieve US county population growth data for a selected set of years. We'll then use this data to calculate population growth rates across multiple periods.

## An introduction to the American Community Survey

Many Americans probably associate the US Census Bureau with its decennial Census. However, the Census Bureau also conducts the [American Community Survey](https://www.census.gov/data/developers/data-sets/acs-5year.html) each year, making it a great resource for recent demographic data.

This notebook will source data from the American Community Survey's 5-year estimates, which show an average of results for the past 5 years. (For example, the 2021 ACS5 dataset shows results between 2017 and 2021). The [1-year ACS estimates](https://www.census.gov/data/developers/data-sets/acs-1year.html) offer results for a more recent timeframe; however, because the 5-year estimates are sourced from a larger pool of data, they may be more reliable (especially for smaller regions). In addition, 1-year estimates aren't available for certain regions, such as zip codes.

For the sake of brevity, I'll often refer to the American Community Survey's 5-year estimates as the 'ACS5' survey.

# Introducing the Census API

In the process of working to answer the first question (regarding where to move to start a family), we'll also explore the Census API's capabilities and craft a Python function that will use it to efficiently retrieve data.

Importing relevant libraries and setting two configuration variables:

In [1]:
import time
program_start_time = time.time()
import pandas as pd
import numpy as np
from iteration_utilities import duplicates
pd.set_option('display.max_columns', 1000)
import lxml # Necessary for reading online HTML tables into Pandas

render_for_pdf = False
if render_for_pdf == True:
    pd.set_option('display.max_columns', 4)


acs5_year = 2021 # By updating this variable when future American 
# Community Surveys get released, you should be able to retrieve the most
# recent data possible. (If changes to the survey's format are made,
# however, updates to the scripts may be necessary.)

# Note: I had originally set acs5_year to 2022, the latest year for which
# ACS5 data were available at the time. However, due to a recent change
# in Connecticut's county-equivalent boundaries (see
# https://www.federalregister.gov/documents/2022/06/06/2022-12063/
# change-to-county-equivalents-in-the-state-of-connecticut for more
# information), ACS5 population growth data between previous
# years and 2022 appeared to be unavailable for that state. Therefore,
# I chose to retrieve data for 2021 instead. 

download_variable_list = True # If set to True, a new list of variables
# will be downloaded from the Census API website. If False, this list of 
# variables will instead be read in from a local .csv copy (thus saving
# processing time).

## Importing a Census API Key

You can obtain a free Census API key [at this website](https://api.census.gov/data/key_signup.html). The following cell imports my own personal key, so you'll need to replace this code with one that loads in your own API key.

In [2]:
with open ('census_api_key_path.txt') as file:
    key_path = file.read()
with open(key_path) as file:
    key = file.read()

The Census offers detailed API documentation that makes retrieving data from it relatively straightforward. For instance, you'll probably find the [Census Data API User Guide](https://www.census.gov/content/dam/Census/data/developers/api-user-guide/api-guide.pdf) to be helpful in applying the Census API.

[This list](https://api.census.gov/data/2021/acs/acs5/examples.html) of ACS5 API call examples is another great resource. One of the sample URLs shown on this page for retrieving county-level data appears as follows:

https://api.census.gov/data/2021/acs/acs5?get=NAME,B01001_001E&for=county:*&key=YOUR_KEY_GOES_HERE

If you replace the 'YOUR_KEY_GOES_HERE' component of the URL with your actual key, then enter this link into your web browser, you'll receive a very long list of counties, population values, and state and county codes. The top of the list for the 2021 ACS5 looks like this:

```
[["NAME","B01001_001E","state","county"],
["Autauga County, Alabama","58239","01","001"],
["Baldwin County, Alabama","227131","01","003"],
["Barbour County, Alabama","25259","01","005"],
["Bibb County, Alabama","22412","01","007"],
["Blount County, Alabama","58884","01","009"],
["Bullock County, Alabama","10386","01","011"],
```

'B01001_001E' refers to the total population estimates for a given county. We can find this out by going to the [2021 ACS5's Detailed Tables page for the](https://api.census.gov/data/2021/acs/acs5/variables.html) and navigating to the row with a 'Name' value of 'B01001_001E'. This link, which may take a little while to fully load, is available on the [ACS5 API Documentation Page](https://www.census.gov/data/developers/data-sets/acs-5year.html).


We can use `pd.read_json()` to easily read this same data into a DataFrame:

In [3]:
df_results = pd.read_json(
    f'https://api.census.gov/data/{acs5_year}/\
acs/acs5?get=NAME,B01001_001E&for=county:*&key={key}')
# read_json documentation:
# https://pandas.pydata.org/pandas-docs/stable/reference/api/
# pandas.read_json.html
df_results.head()

,0,1,2,3
0,NAME,B01001_001E,state,county
1,"Autauga County, Alabama",58239,01,001
2,"Baldwin County, Alabama",227131,01,003
3,"Barbour County, Alabama",25259,01,005
4,"Bibb County, Alabama",22412,01,007


At this point, the DataFrame's columns are [0, 1, 2, 3], whereas the columns we want to use are stored within the first row. The following code sets these row values as our column values, then deletes this row:

In [4]:
df_results.columns = df_results.iloc[0]
df_results.drop(0, inplace = True)

df_results.head()

,NAME,B01001_001E,state,county
1,"Autauga County, Alabama",58239,01,001
2,"Baldwin County, Alabama",227131,01,003
3,"Barbour County, Alabama",25259,01,005
4,"Bibb County, Alabama",22412,01,007
5,"Blount County, Alabama",58884,01,009


## Retrieving our Data

In order to determine which variable codes to enter into our script, we'll first need to review a list of all American Community Survey variables and the overall groups into which they fit. This list is available on the Census website ([here's the copy for 2021](https://api.census.gov/data/2021/acs/acs5/variables.html)), but we can also use Pandas to import them into a DataFrame, as shown below.

In [5]:
if download_variable_list == True:
    df_variables_page = pd.read_html(
        f'https://api.census.gov/data/{acs5_year}/acs/acs5/variables.html')[0] 
    # [0] selects the first HTML table found on this page.
    # See https://pandas.pydata.org/pandas-docs/stable/reference/api/
    # pandas.read_html.html
    # for more information on pd.read_html().
        
    # Some rows in this table contain items other than demographic 
    # variables (e.g. region names). We can exclude them by selecting 
    # only rows that begin with 'Estimate'. (Another option would have 
    # been to filter out rows with N/A 'Group' entries (i.e. 
    # df_variables.query("Group.isna() == False")), 
    # but this would have left a couple non-variable rows in place.
    
    df_variables = df_variables_page[
    df_variables_page['Label'].str[0:8] == 'Estimate'].copy(
    ).reset_index(drop=True)
    # Saving this table to a local .csv file:
    df_variables.to_csv(f'Datasets/{acs5_year}_variables.csv', 
    index = False)
else: # Reading a local copy of this dataset instead, which should 
    # take much less time. 
    df_variables = pd.read_csv(
        f'Datasets/{acs5_year}_variables.csv')
df_variables.head()

,Name,Label,Concept,Required,Attributes,Limit,Predicate Type,Group,Unnamed: 8
0,B01001_001E,Estimate!!Total:,SEX BY AGE,not required,"B01001_001EA, B01001_001M, B01001_001MA",0,int,B01001,NaN
1,B01001_002E,Estimate!!Total:!!Male:,SEX BY AGE,not required,"B01001_002EA, B01001_002M, B01001_002MA",0,int,B01001,NaN
2,B01001_003E,Estimate!!Total:!!Male:!!Under 5 years,SEX BY AGE,not required,"B01001_003EA, B01001_003M, B01001_003MA",0,int,B01001,NaN
3,B01001_004E,Estimate!!Total:!!Male:!!5 to 9 years,SEX BY AGE,not required,"B01001_004EA, B01001_004M, B01001_004MA",0,int,B01001,NaN
4,B01001_005E,Estimate!!Total:!!Male:!!10 to 14 years,SEX BY AGE,not required,"B01001_005EA, B01001_005M, B01001_005MA",0,int,B01001,NaN


With over 28,000 individual variables, it could take a very long time to identify the items you'd like to retrieve from the Census. We can make this search process somewhat easier by creating a separate *groups* table that shows only unique group names and their written descriptions (e.g. 'Sex by Age').

In [6]:
df_groups = df_variables.drop_duplicates(
    'Group')[['Concept', 'Group']].copy(
    ).reset_index(drop=True)
df_groups.head()

,Concept,Group
0,SEX BY AGE,B01001
1,SEX BY AGE (WHITE ALONE),B01001A
2,SEX BY AGE (BLACK OR AFRICAN AMERICAN ALONE),B01001B
3,SEX BY AGE (AMERICAN INDIAN AND ALASKA NATIVE ...,B01001C
4,SEX BY AGE (ASIAN ALONE),B01001D


We'll save this group table to a local .csv file as well:

In [7]:
df_groups.to_csv(f'Datasets/{acs5_year}_groups.csv', 
                 index = False)

In order to find variables of interest, I recommend first searching for keywords of interest within the group table (which is much smaller in size) in order to identify relevant group IDs. Next, you can search for those group IDs inside the variables table in order to find the exact metrics to request from the Census API.

The following table stores variables for three separate groups: (1) the total population; (2) all males aged 25 to 29 years; and (3) all females aged 25 to 29 years. (The B01001 table that stores these values didn't have an entry for all people aged 25 to 29; we'll get around this limitation by retrieving sex-specific population totals within this age group, then adding those totals together.)

In [8]:
grad_destinations_variable_list = [
    'B01001_001E', 'B01001_011E',
    'B01001_035E']

The demographic columns in the Census API's output are labeled with their variable names (e.g. 'B01001_001E'). These names are concise, but you'll need a copy of the original variable list to interpret them. Therefore, I chose to replace these column names with a combination of the 'Label', 'Concept', and 'Name' entries in the original variable list. These column names are very long, but they do make the output easier to interpret (while also preserving the original names for reference). 

In addition, if the description corresponding to a variable name happens to change from one year to another, the use of aliases will help you identify that change. (This will help prevent you from treating two different data types that happened to use the same variable code in different years as equal. However, I would imagine that the Census wouldn't repurpose variable codes in this way.)

The following function assists with this replacement by creating a dictionary whose keys are the original field names (e.g. 'B0101_001E') and whose values are the replacement names (e.g. 'Sex by Age_Estimate!!Total:_B01001_001E').

In [9]:
def create_variable_aliases(df_variables, variable_list):
    '''This function creates a dictionary whose keys are 
    the original 'Name' values (e.g. 'B001_001E') within a variable
    list on the Census API website and whose values are the replacement 
    names (e.g. 'Sex by Age_Estimate!!Total:_B01001_001E').
    This resulting dictionary can then be passed to a df.rename() call
    within retrieve_census_data() in order to make the output of that
    function easier to interpret.
    
    df_variables: A DataFrame containing a list of Census variables. For
    an example of this list for the 2021 American Community Survey (5-Year 
    Estimates), visit: 
    https://api.census.gov/data/2021/acs/acs5/examples.html .
    
    variable_list: The list of variables to rename 
    (e.g. ['B01001_001E', 'B01001_002E']).
    '''
    # Creating a DataFrame that contains the information needed for the
    # updated column names:
    df_aliases = df_variables.query(
        "Name in @variable_list")[['Name', 'Label', 'Concept']].copy()
    # Creating a new 'Description' column that will replace the original
    # output field names:
    df_aliases['Description'] = (df_aliases['Concept'] 
                                 + '_' + df_aliases['Label'] 
                                 + ' (' + df_aliases['Name'] + ')')
    # Creating a dictionary whose keys are the original field names and 
    # whose values are the new 'Description' entries that were 
    # just created:
    alias_dict = df_aliases.set_index('Name').to_dict()['Description']
    # See https://pandas.pydata.org/pandas-docs/stable/reference/api/
    # pandas.DataFrame.to_dict.html
    return alias_dict

Creating our aliases:

In [10]:
grad_destinations_alias_dict = create_variable_aliases(
    df_variables = df_variables, 
    variable_list = grad_destinations_variable_list)
grad_destinations_alias_dict

{'B01001_001E': 'SEX BY AGE_Estimate!!Total: (B01001_001E)',
 'B01001_011E': 'SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)',
 'B01001_035E': 'SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'}

## Defining a Census data retrieval function

The following function simplifies the process of retrieving data from the Census API. It also enables the user to rename variable fields (e.g. 'B01001_001E') with aliases for those fields (e.g. 'Sex by Age_Estimate!!Total: (B01001_001E)'), but this option is disabled by default. In addition, it allows more than 50 variables to be retrieved at the same time, thus making it easier to retrieve especially large datasets.

[Note: currently, this function only supports data retrieval for the ACS 5-year and 1-year estimates. However, I may add in the ability to retrieve decennial Census data in the future.]

In [11]:
def retrieve_census_data(survey, year, region, key, variable_list,
                         rename_data_fields = False, 
                         field_vars_dict = {}):
    '''This function (which I plan to expand) retrieves data from the US
    Census API. It accommodates more than 50 variables.
    
    survey: the survey from which to retrieve data. The only arguments
    currently supported are 'acs5' and 'acs1' (for the American Community 
    Survey 5-Year and 1-Year estimates, respectively).
    
    year: the year for which you wish to retrieve survey data. Note that,
    When region is set to 'acs5', the survey results will include data
    for the 5 years leading up to (and including) the 'year' argument.
    (For example, if you set 'year' to 2021, you'll retrieve ACS5 data
    from 2017 to 2021 (inclusive).)
    
    
    region: The geographic level at which you wish to retrieve data. 
    Examples include 'us', 'state', 'county', 'zip', 'msa' 
    (for metropolitan/micropolitan statistical area data), and 'csa' 
    (for combined statistical area data); 
    however, other regions are supported as well. Consult your survey's 
    API examples page for other options. (For instance, if you wanted to 
    retrieve data by urban area within the 2021 ACS5, you could go to 
    https://api.census.gov/data/2021/acs/acs5/examples.html, then search
    for 'urban area.' The Urban Area URL ends with
    '&for=urban%20area:*&key=YOUR_KEY_GOES_HERE'. Therefore, you'd want to
    use 'urban%20area' as your 'region' argument.)   

    (Note: 'zip' will retrieve results by Zip Code
    Tabulation Area, which are similar to (but not identical to)
    # zip codes. See 
    # https://en.wikipedia.org/wiki/ZIP_Code_Tabulation_Area
    # for more information.
    
    variable_list: The list of variables for which to retrieve data.

    key: your personal Census API key.

    rename_data_fields: set to True to replace column names in your 
    dataset with new entries of your choice.

    field_vars_dict: A dictionary that stores the original variable names
    retrieved by the Census (e.g. 'B01001_001E' as keys and your desired
    replacements as values. Example: 
    {'B01001_001E': 'Sex by Age_Estimate!!Total:_B01001_001E',
     'B01001_002E': 'Sex by Age_Estimate!!Total:!!Male:_B01001_002E'}'
     
    '''

    # Using the iteration_utilities library to check for duplicate
    # values within variable_list (which could cause issues later on):
    # The following code is based on
    # https://iteration-utilities.readthedocs.io/en/latest/generated/
    # duplicates.html
    duplicate_variables = list(duplicates(variable_list))
    
    if len(duplicate_variables) > 0:
        raise ValueError(f"The following variables appear more than once \
in your variable list: {duplicate_variables}")
    
    if survey == 'acs5':
        survey_string = 'acs/acs5'

    elif survey == 'acs1':
        survey_string = 'acs/acs1'
    
    else:
        raise ValueError("This survey type is not currently supported by \
                         the function.")

    
    # Converting simplified region names into strings that the API 
    # function will recognize:
    if region == 'zip':
        region = 'zip%20code%20tabulation%20area' # Based on
        # the ZCTA example within
        # https://api.census.gov/data/2021/acs/acs5/examples.html
    
    if region == 'csa':
        region = 'combined%20statistical%20area'
    
    if region == 'msa':
        region = 'metropolitan%20statistical\
%20area/micropolitan%20statistical%20area'

    
    # Only 50 variables can be retrieved from the Census API at a time 
    # using the approach shown in this function. The following code 
    # accommodates this limitation by splitting variable_list into 
    # sublists of up to 49 variables. The data retrieved for the variables 
    # in these sublists will then get merged back together.
    # (49 variables are retrieved at a time instead of 50 because it 
    # appears that the initial 'NAME' variable also counts towards 
    # the 50-variable limit.)
    
    i = 0
       
    while i < len(variable_list): # i.e. while there
        # are still more variables to iterate through
        variable_sublist = variable_list[i:i+49] # This line reads the 
        # next 49 variables from variable_list into a sublist that can 
        # then be\ passed to the API
        # print("variable_sublist:", variable_sublist)
        # Converting the list of variables into a string that can be 
        # passed to the API call:
        # (The Census API guide at
        # https://www.census.gov/content/dam/Census/data/developers/
        # api-user-guide/api-guide.pdf
        # demonstrates how to call multiple census variables at once.)
        variable_string = ','.join(variable_sublist)
        # print("variable_string:",variable_string)
    
        # Retrieving data via the Census API:
        # This line was originally based on an example found in
        # https://api.census.gov/data/2022/acs/acs5/examples.html .
    
        # read_json documentation:
        # https://pandas.pydata.org/pandas-docs/stable/reference/api/
        # pandas.read_json.html

        api_url = f'https://api.census.gov/data/{year}/\
{survey_string}?get=NAME,{variable_string}&for={region}:*&key={key}'
        # print(api_url)
        
        df_results = pd.read_json(api_url)
    
        # At this point, the DataFrame's columns are a list of integers; 
        # the desired column names are stored within the first row. 
        # The following code resolves this issue by setting these row 
        # values as the column values and then deleting this row.
    
        df_results.columns = df_results.iloc[0]
        df_results.drop(0, inplace = True)


        # Determining which merge keys to use when combining API results
        # for different sublists together:
        # This is made more complicated by the fact that results for 
        # different regions will have different identifier
        # columns (e.g. 'NAME', 'county', and 'state' for county data but 
        # only 'NAME' and 'state' for state data). However, we can 
        # accommodate this behavior by simply initializing our list of 
        # merge keys as the set of all columns that are *not* also 
        # variable columns.
        if i == 0: # This step only needs to be performed for our first
            # sublist of variables, since merge keys for other sublists
            # will be identical.
            merge_keys = list(set(df_results.columns) 
              - set(variable_sublist))
            # print("merge_keys:",merge_keys)

        if i == 0: # Since this is the first set 
            # of results, we can initialize df_combined_results 
            # as a copy of df_results.
            df_combined_results = df_results.copy()
        else: # Merging our latest set of results into df_results:
            df_combined_results = df_combined_results.merge(
                df_results, on = merge_keys,
                how = 'outer').copy()
            # Added .copy() here in response to a data fragmentation 
        # warning

        i += 49 
        # Allows the function to iterate through the next 49 variables
        # within variable_list

        
    # Converting variable columns to numeric data types:
    for column in variable_list:
        # print(f"Now converting {column} to a numeric type.")
        df_combined_results[column] = pd.to_numeric(
            df_combined_results[column])
        # pd.to_numeric() allows for either integer or float outputs
        # depending on the nature of the original data.
        # See https://pandas.pydata.org/pandas-docs/stable/reference/api/
        # pandas.to_numeric.html

    # Replacing column names with aliases if requested:
    if rename_data_fields == True:
        df_combined_results.rename(
            columns = field_vars_dict, inplace = True)

    # The following for loop moves all of the merge keys (e.g. geographic
    # identifiers) to the left side of the table. This is particularly
    # useful when retrieving longer lists of variables, as otherwise,
    # certain keys can get buried in the middle of the dataset
    for i in range(len(merge_keys)):
        df_combined_results.insert(
            i, merge_keys[i], 
            df_combined_results.pop(merge_keys[i]))

    # Adding a 'Year' column to the left of all existing DataFrame columns:
    # (this will prove particularly
    # helpful when comparing data from different years.)
    df_combined_results.insert(0, 'Year', year)
    
    return df_combined_results

(The following code allowed me to test out retrieve_census_data for a particularly long variable list.)

In [12]:
# test_list = list(df_variables['Name'][0:151])

# test_alias_dict = create_variable_aliases(
#     df_variables = df_variables, 
#     variable_list = test_list)

# test_acs5_data = retrieve_census_data(
#     survey = 'acs5', year = acs5_year, region = 'county',
#     variable_list = test_list, 
#     rename_data_fields = True, 
#     field_vars_dict = test_alias_dict, key = key)

# test_acs5_data

Next, we'll define a list of years for which we would like to retrieve Census data. In order to make this code easier to use in future years, I'll define these years as an offset of acs5_year rather than hardcoding them.

In [13]:
years_to_retrieve = [acs5_year - 12, acs5_year -10, 
                     acs5_year - 8,
                     acs5_year - 6,
                     acs5_year - 5,
                     acs5_year]
# American Community Survey 1-year estimates weren't available in 2020,
# so you'll want to remove that year from your list if it happens to be present.
# However, because I decided to use 5-year rather than 1-year estimates,
# I commented out this line.
# if 2020 in years_to_retrieve:
#     years_to_retrieve.remove(2020)
years_to_retrieve

[2009, 2011, 2013, 2015, 2016, 2021]

At this point, it's a good idea to confirm that our three variable codes ('B01001_001E', 'B01001_011E', and 'B01001_035E') had the same meaning for all the years whose data we'll be retrieving. We can do so by running the following code, which retrieves these variables and their corresponding descriptions for all of the years in years_to_retrieve.

(Due to the size of the variables.html page, this code can take a while to run, so I commented it out below.)

In [14]:
# var_meanings_by_year_df_list = []
# for year in years_to_retrieve:
#     df_var_list = pd.read_html(
#         f'https://api.census.gov/data/{year}/acs/acs5/variables.html')[
#     0][['Name', 'Label', 'Concept']].query(
#         "Name in @grad_destinations_variable_list")
#     df_var_list.insert(0, 'Year', year)
#     var_meanings_by_year_df_list.append(df_var_list)
# df_var_meanings_by_year = pd.concat([df for df in var_meanings_by_year_df_list])
# df_var_meanings_by_year.to_csv('var_meanings_by_year.csv', index = False)
# df_var_meanings_by_year

Here's a saved .csv copy of this table that confirms that these codes had the same meaning in each of the years we're analyzing:

In [15]:
pd.read_csv('var_meanings_by_year.csv')

,Year,Name,Label,Concept
0,2010,B01001_001E,Estimate!!Total,SEX BY AGE
1,2010,B01001_011E,Estimate!!Total!!Male!!25 to 29 years,SEX BY AGE
2,2010,B01001_035E,Estimate!!Total!!Female!!25 to 29 years,SEX BY AGE
3,2012,B01001_001E,Estimate!!Total,SEX BY AGE
4,2012,B01001_011E,Estimate!!Total!!Male!!25 to 29 years,SEX BY AGE
5,2012,B01001_035E,Estimate!!Total!!Female!!25 to 29 years,SEX BY AGE
6,2014,B01001_001E,Estimate!!Total,SEX BY AGE
7,2014,B01001_011E,Estimate!!Total!!Male!!25 to 29 years,SEX BY AGE
8,2014,B01001_035E,Estimate!!Total!!Female!!25 to 29 years,SEX BY AGE
9,2016,B01001_001E,Estimate!!Total,SEX BY AGE


We're now ready to retrieve our population totals for the years referenced in years_to_retrieve. We'll do so by calling retrieve_census_data() for each of these years via a for loop, then adding their respective DataFrames together using pd.concat().

(Note: Only 5825 results showed up when I requested ACS1 data (as opposed to over 25,000 results for ACS5 data), so many counties were not getting incorporated within the ACS1 results. Therefore, the ACS5 should be the best survey to use for evaluating county-level growth.)

In [16]:
census_data_by_year_df_list = []
for year in years_to_retrieve:
    df_data = retrieve_census_data(
        survey = 'acs5', year = year, 
        region = 'county',
        variable_list = grad_destinations_variable_list, 
        rename_data_fields = True, field_vars_dict = grad_destinations_alias_dict,
        key = key)
    census_data_by_year_df_list.append(df_data)
df_growth_data_by_year = pd.concat(df for df in census_data_by_year_df_list)
# Removing Puerto Rico from our list of results so as to focus only on counties
# and county equivalents within the 50 US states + DC:
df_growth_data_by_year.query("state != '72'", inplace = True)
df_growth_data_by_year

,Year,county,NAME,state,SEX BY AGE_Estimate!!Total: (B01001_001E),SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E),SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)
1,2009,091,"Dodge County, Georgia",13,19695,634,574
2,2009,093,"Dooly County, Georgia",13,11641,531,352
3,2009,095,"Dougherty County, Georgia",13,95330,3264,3900
4,2009,097,"Douglas County, Georgia",13,122657,4066,4353
5,2009,099,"Early County, Georgia",13,11812,275,282
...,...,...,...,...,...,...,...
3139,2021,037,"Sweetwater County, Wyoming",56,42459,1320,1259
3140,2021,039,"Teton County, Wyoming",56,23319,1142,905
3141,2021,041,"Uinta County, Wyoming",56,20514,574,567
3142,2021,043,"Washakie County, Wyoming",56,7768,175,200


The following cell adds together male and female population totals in order to calculate the total number of 25- to 29-year-olds within each county. It also simplifies the original total population column name in order to improve its readability.

In [17]:
df_growth_data_by_year['Total_Pop_25_to_29'] = (df_growth_data_by_year[
'SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)'] + 
df_growth_data_by_year[
'SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'])
df_growth_data_by_year.rename(
    columns = {'SEX BY AGE_Estimate!!Total: (B01001_001E)':'Total_Pop'}, 
inplace = True)
df_growth_data_by_year.drop(
    ['SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)',
     'SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'],
     axis = 1, inplace = True)
                             

df_growth_data_by_year

,Year,county,NAME,state,Total_Pop,Total_Pop_25_to_29
1,2009,091,"Dodge County, Georgia",13,19695,1208
2,2009,093,"Dooly County, Georgia",13,11641,883
3,2009,095,"Dougherty County, Georgia",13,95330,7164
4,2009,097,"Douglas County, Georgia",13,122657,8419
5,2009,099,"Early County, Georgia",13,11812,557
...,...,...,...,...,...,...
3139,2021,037,"Sweetwater County, Wyoming",56,42459,2579
3140,2021,039,"Teton County, Wyoming",56,23319,2047
3141,2021,041,"Uinta County, Wyoming",56,20514,1141
3142,2021,043,"Washakie County, Wyoming",56,7768,375


The following code applies pd.pivot() to place all population totals for a given county within the same row, thus making future growth calculations easier.

In [18]:
df_growth_data_by_year_pivot = df_growth_data_by_year.copy().pivot(
    columns = 'Year', index = ['NAME', 'county', 'state']).reset_index()
# The values could be named explicitly, but since pivot() will infer them
    # automatically, there's no need to do so. The following code is thus
    # commented out.
    # values = ['Total_Pop',
#           'Total_Pop_25_to_29']

df_growth_data_by_year_pivot.head()

0                                 NAME county state Total_Pop            \
Year                                                     2009      2011   
0     Abbeville County, South Carolina    001    45   25347.0   25515.0   
1             Acadia Parish, Louisiana    001    22   59616.0   61430.0   
2            Accomack County, Virginia    001    51   38522.0   33701.0   
3                    Ada County, Idaho    001    16  368791.0  388174.0   
4                   Adair County, Iowa    001    19    7555.0    7699.0   

0                                            Total_Pop_25_to_29           \
Year      2013      2015      2016      2021               2009     2011   
0      25233.0   24997.0   24951.0   24374.0             1463.0   1239.0   
1      61847.0   62163.0   62372.0   58200.0             4123.0   4217.0   
2      33289.0   33115.0   33060.0   33388.0             2469.0   1842.0   
3     401673.0  417501.0  425798.0  485246.0            32200.0  29327.0   
4       7588.0    7426.0    7330.0    7439.0              284.0    425.0   

0                                         
Year     2013     2015     2016     2021  
0      1118.0   1191.0   1226.0   1292.0  
1      4172.0   4126.0   4240.0   3705.0  
2      1810.0   1669.0   1819.0   1907.0  
3     29494.0  30142.0  30646.0  33625.0  
4       392.0    362.0    362.0    411.0

We'll now call to_flat_index() in order to group all column data within the same row:

In [19]:
df_growth_data_by_year_pivot.columns = (
    df_growth_data_by_year_pivot.columns.to_flat_index())

df_growth_data_by_year_pivot.head()

,"(NAME, )","(county, )","(state, )","(Total_Pop, 2009)","(Total_Pop, 2011)","(Total_Pop, 2013)","(Total_Pop, 2015)","(Total_Pop, 2016)","(Total_Pop, 2021)","(Total_Pop_25_to_29, 2009)","(Total_Pop_25_to_29, 2011)","(Total_Pop_25_to_29, 2013)","(Total_Pop_25_to_29, 2015)","(Total_Pop_25_to_29, 2016)","(Total_Pop_25_to_29, 2021)"
0,"Abbeville County, South Carolina",001,45,25347.0,25515.0,25233.0,24997.0,24951.0,24374.0,1463.0,1239.0,1118.0,1191.0,1226.0,1292.0
1,"Acadia Parish, Louisiana",001,22,59616.0,61430.0,61847.0,62163.0,62372.0,58200.0,4123.0,4217.0,4172.0,4126.0,4240.0,3705.0
2,"Accomack County, Virginia",001,51,38522.0,33701.0,33289.0,33115.0,33060.0,33388.0,2469.0,1842.0,1810.0,1669.0,1819.0,1907.0
3,"Ada County, Idaho",001,16,368791.0,388174.0,401673.0,417501.0,425798.0,485246.0,32200.0,29327.0,29494.0,30142.0,30646.0,33625.0
4,"Adair County, Iowa",001,19,7555.0,7699.0,7588.0,7426.0,7330.0,7439.0,284.0,425.0,392.0,362.0,362.0,411.0


Next, we'll convert the tuple-based columns created by to_flat_index() to string-based ones by executing a list comprehension. This list comprehension will convert tuples that contain both a string and an integer (e.g. ('Total_Pop', 2011)) to a single string with an underscore separating the two elements. Tuples with an empty second entry (e.g. ('county', '') will get replaced with just the first entry (e.g. 'county').

In [20]:
df_growth_data_by_year_pivot.columns = [
    column[0] + '_' + str(column[1]) if isinstance(column[1], int) 
    else column[0] for column in df_growth_data_by_year_pivot.columns]
df_growth_data_by_year_pivot.head()

,NAME,county,state,Total_Pop_2009,Total_Pop_2011,Total_Pop_2013,Total_Pop_2015,Total_Pop_2016,Total_Pop_2021,Total_Pop_25_to_29_2009,Total_Pop_25_to_29_2011,Total_Pop_25_to_29_2013,Total_Pop_25_to_29_2015,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2021
0,"Abbeville County, South Carolina",001,45,25347.0,25515.0,25233.0,24997.0,24951.0,24374.0,1463.0,1239.0,1118.0,1191.0,1226.0,1292.0
1,"Acadia Parish, Louisiana",001,22,59616.0,61430.0,61847.0,62163.0,62372.0,58200.0,4123.0,4217.0,4172.0,4126.0,4240.0,3705.0
2,"Accomack County, Virginia",001,51,38522.0,33701.0,33289.0,33115.0,33060.0,33388.0,2469.0,1842.0,1810.0,1669.0,1819.0,1907.0
3,"Ada County, Idaho",001,16,368791.0,388174.0,401673.0,417501.0,425798.0,485246.0,32200.0,29327.0,29494.0,30142.0,30646.0,33625.0
4,"Adair County, Iowa",001,19,7555.0,7699.0,7588.0,7426.0,7330.0,7439.0,284.0,425.0,392.0,362.0,362.0,411.0


## Adding in comparison fields

Now that we have population data for several years in a relatively easy-to-parse format, we can create comparisons between those years--along with corresponding percentile and rank information.

The following code was derived from my create_enrollment_comparisons() function within  catholic_school_enrollment_trends_v4.ipynb (available at https://github.com/kburchfiel/catholic_school_enrollment/blob/main/catholic_school_enrollment_trends_v4.ipynb).

In [21]:
def create_comparison_fields(df, field_var, year_list,
                             field_year_separator = '_'):
    '''This function calculates nominal and percentage changes
    between the last year in a list and all years leading up to that year.
    It also calculates both rank and percentile information for these
    changes.
    
    df: the DataFrame that will be updated with comparisons between years.
    This function assumes that the fields within df that contain 
    data for a particular year use the format
    {field_var}{field_year_separator}{latest_year} (e.g. 'Total_Pop_2009', 
    'Total_Pop_2015', etc.). 
    
    field_var: A string representing the variable 
    whose values should be compared (e.g. 'Total_Pop' within the field
    'Total_Pop_2009'. 

    year_list: A list of years to compare. The function will compare
    all years leading up to the last year with the last year. For example,
    if year_list equals [2005, 2009, 2015], the function will create
    comparisons between (1) 2005 and 2015 and (2) 2009 and 2015 (but not
    2005 and 2009).
    field_var data for each of these years should be stored within
    the DataFrame. For instance, if year_list is equal to the example
    shown above, field_var is 'Total_Pop', and field_year_separator (see
    below) is '_', the script will expect to see the following fields
    within the DataFrame:
    'Total_Pop_2005', 'Total_Pop_2009', 'Total_Pop_2015'

    field_year_separator: The character (e.g. a space, an underscore,
    etc.) separating the field_var and year values within the DataFrame's
    fields.
    
    '''
    latest_year = year_list[-1]
    for year in year_list[:-1]: # E.g. for all years leading up 
        # to (but not including) latest_year

        # Calculating the nominal change between the two years:
        df[f'{year}-{latest_year} {field_var} Change'] = (
        df[f'{field_var}{field_year_separator}{latest_year}'] 
        - df[f'{field_var}{field_year_separator}{year}'] )

        # Calculating the percentage change:
        df[f'{year}-{latest_year} {field_var} % Change'] = 100*((
        df[f'{field_var}{field_year_separator}{latest_year}'] 
        / df[f'{field_var}{field_year_separator}{year}']) - 1)

    
        # Calculating ranks and percentiles for both the nominal 
        # change and % change columns:
        # Note that field_var still needs to be included within these
        # columns in order to specify what change, exactly, is being
        # analyzed. (This is particularly important when this function
        # gets called for multiple field_var entries.)
        df[f'{year}-{latest_year} {field_var} Change Rank'] = df[
        f'{year}-{latest_year} {field_var} Change'].rank(
            ascending=False, method = 'min')
        
        df[f'{year}-{latest_year} {field_var} Change Percentile'] = (
        100 * df[
        f'{year}-{latest_year} {field_var} Change'].rank(
            pct=True, ascending=True, method='max'))
        
        df[f'{year}-{latest_year} {field_var} % Change Rank'] = (
            df[f'{year}-{latest_year} {field_var} % Change'].rank(
            ascending=False, method = 'min'))
        
        df[f'{year}-{latest_year} {field_var} % Change Percentile'] = (
            100 * df[
            f'{year}-{latest_year} {field_var} % Change'].rank(
            pct=True, ascending=True, method='max'))

`create_comparison_fields` will now be called twice to create growth metrics for both total county populations and residents in the 25-29 age range.

In [22]:
for field_var in ['Total_Pop', 'Total_Pop_25_to_29']:
    create_comparison_fields(
        df = df_growth_data_by_year_pivot,
        field_var = field_var,
        year_list = years_to_retrieve,
        field_year_separator='_')

In [23]:
df_growth_data_by_year_pivot.head(5)

,NAME,county,state,Total_Pop_2009,Total_Pop_2011,Total_Pop_2013,Total_Pop_2015,Total_Pop_2016,Total_Pop_2021,Total_Pop_25_to_29_2009,Total_Pop_25_to_29_2011,Total_Pop_25_to_29_2013,Total_Pop_25_to_29_2015,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2021,2009-2021 Total_Pop Change,2009-2021 Total_Pop % Change,2009-2021 Total_Pop Change Rank,2009-2021 Total_Pop Change Percentile,2009-2021 Total_Pop % Change Rank,2009-2021 Total_Pop % Change Percentile,2011-2021 Total_Pop Change,2011-2021 Total_Pop % Change,2011-2021 Total_Pop Change Rank,2011-2021 Total_Pop Change Percentile,2011-2021 Total_Pop % Change Rank,2011-2021 Total_Pop % Change Percentile,2013-2021 Total_Pop Change,2013-2021 Total_Pop % Change,2013-2021 Total_Pop Change Rank,2013-2021 Total_Pop Change Percentile,2013-2021 Total_Pop % Change Rank,2013-2021 Total_Pop % Change Percentile,2015-2021 Total_Pop Change,2015-2021 Total_Pop % Change,2015-2021 Total_Pop Change Rank,2015-2021 Total_Pop Change Percentile,2015-2021 Total_Pop % Change Rank,2015-2021 Total_Pop % Change Percentile,2016-2021 Total_Pop Change,2016-2021 Total_Pop % Change,2016-2021 Total_Pop Change Rank,2016-2021 Total_Pop Change Percentile,2016-2021 Total_Pop % Change Rank,2016-2021 Total_Pop % Change Percentile,2009-2021 Total_Pop_25_to_29 Change,2009-2021 Total_Pop_25_to_29 % Change,2009-2021 Total_Pop_25_to_29 Change Rank,2009-2021 Total_Pop_25_to_29 Change Percentile,2009-2021 Total_Pop_25_to_29 % Change Rank,2009-2021 Total_Pop_25_to_29 % Change Percentile,2011-2021 Total_Pop_25_to_29 Change,2011-2021 Total_Pop_25_to_29 % Change,2011-2021 Total_Pop_25_to_29 Change Rank,2011-2021 Total_Pop_25_to_29 Change Percentile,2011-2021 Total_Pop_25_to_29 % Change Rank,2011-2021 Total_Pop_25_to_29 % Change Percentile,2013-2021 Total_Pop_25_to_29 Change,2013-2021 Total_Pop_25_to_29 % Change,2013-2021 Total_Pop_25_to_29 Change Rank,2013-2021 Total_Pop_25_to_29 Change Percentile,2013-2021 Total_Pop_25_to_29 % Change Rank,2013-2021 Total_Pop_25_to_29 % Change Percentile,2015-2021 Total_Pop_25_to_29 Change,2015-2021 Total_Pop_25_to_29 % Change,2015-2021 Total_Pop_25_to_29 Change Rank,2015-2021 Total_Pop_25_to_29 Change Percentile,2015-2021 Total_Pop_25_to_29 % Change Rank,2015-2021 Total_Pop_25_to_29 % Change Percentile,2016-2021 Total_Pop_25_to_29 Change,2016-2021 Total_Pop_25_to_29 % Change,2016-2021 Total_Pop_25_to_29 Change Rank,2016-2021 Total_Pop_25_to_29 Change Percentile,2016-2021 Total_Pop_25_to_29 % Change Rank,2016-2021 Total_Pop_25_to_29 % Change Percentile
0,"Abbeville County, South Carolina",001,45,25347.0,25515.0,25233.0,24997.0,24951.0,24374.0,1463.0,1239.0,1118.0,1191.0,1226.0,1292.0,-973.0,-3.838719,2627.0,16.262755,2367.0,24.553571,-1141.0,-4.471879,2625.0,16.353204,2309.0,26.426522,-859.0,-3.404272,2521.0,19.694073,2204.0,29.796048,-623.0,-2.492299,2451.0,21.999363,2138.0,31.964343,-577.0,-2.312533,2482.0,21.012416,2174.0,30.818211,-171.0,-11.688312,2629.0,16.198980,2611.0,16.772959,53.0,4.277643,1523.0,51.482308,1557.0,50.398470,174.0,15.563506,987.0,68.578713,617.0,80.369662,101.0,8.480269,1230.0,60.872334,1119.0,64.406240,66.0,5.383361,1397.0,55.555556,1357.0,56.829035
1,"Acadia Parish, Louisiana",001,22,59616.0,61430.0,61847.0,62163.0,62372.0,58200.0,4123.0,4217.0,4172.0,4126.0,4240.0,3705.0,-1416.0,-2.375201,2789.0,11.096939,2158.0,31.218112,-3230.0,-5.258017,3018.0,3.825311,2451.0,21.899904,-3647.0,-5.896810,3049.0,2.868069,2615.0,16.698534,-3963.0,-6.375175,3083.0,1.878383,2760.0,12.161732,-4172.0,-6.688899,3102.0,1.273480,2837.0,9.710283,-418.0,-10.138249,2936.0,6.409439,2544.0,18.909439,-512.0,-12.141333,3083.0,1.753267,2749.0,12.400383,-467.0,-11.193672,3074.0,2.071383,2748.0,12.460166,-421.0,-10.203587,3042.0,3.183699,2747.0,12.575613,-535.0,-12.617925,3055.0,2.769819,2812.0,10.506208
2,"Accomack County, Virginia",001,51,38522.0,33701.0,33289.0,33115.0,33060.0,33388.0,2469.0,1842.0,1810.0,1669.0,1819.0,1907.0,-5134.0,-13.327449,3091.0,1.466837,3032.0,3.348214,-313.0,-0.928756,1968.0,37.2

## Ranking counties by population growth

We'l now sort this dataset to answer a few questions:

1. Which counties had the highest (and lowest) total population growth rates between 2016 and 2021? (We'll evaluate this question for both counties with at least 100,000 residents in 2016 and those with at least one million residents).

2. Which counties experienced the highest (and lowest) 25-to-29-year-old population growth rates?

First, we'll specify a few variables that will be used within our analyses:

In [24]:
range_for_sorting = 5
range_start_year = acs5_year - range_for_sorting 

total_pop_var_for_sorting = 'Total_Pop'
total_pop_sort_column = f'{range_start_year}-{acs5_year} \
{total_pop_var_for_sorting} % Change Rank'

young_pop_var_for_sorting = 'Total_Pop_25_to_29'
young_pop_sort_column = f'{range_start_year}-{acs5_year} \
{young_pop_var_for_sorting} % Change Rank'



In [25]:
total_pop_col_root = f'{range_start_year}-{acs5_year} \
{total_pop_var_for_sorting}' # The 'root' from which other column entries will 
# derive.

young_pop_col_root = f'{range_start_year}-{acs5_year} \
{young_pop_var_for_sorting}'

The following function will help condense the output of our analyses, thus making them more readable.

In [26]:
def gen_display_cols(col_root):
    '''This function specifies which DataFrame columns to display within a 
    given output. It does so by adding various suffixes to a col_root
    value (e.g. '2016-2021 Total_Pop'.'''
    return ['NAME', f'{col_root} Change', f'{col_root} % Change',
            f'{col_root} % Change Rank', f'{col_root} % Change Percentile']

In [27]:
total_pop_display_cols = gen_display_cols(total_pop_col_root)
young_pop_display_cols = gen_display_cols(young_pop_col_root)
total_pop_display_cols

['NAME',
 '2016-2021 Total_Pop Change',
 '2016-2021 Total_Pop % Change',
 '2016-2021 Total_Pop % Change Rank',
 '2016-2021 Total_Pop % Change Percentile']

### Sorting counties with at least 100K residents in 2016 by their 2016-2021 total population growth rates:

In [28]:
df_growth_data_by_year_pivot.query(
    f"Total_Pop_{range_start_year} >= 100000").sort_values(
    total_pop_sort_column).dropna(subset = total_pop_sort_column)[
total_pop_display_cols].reset_index(drop=True)

,NAME,2016-2021 Total_Pop Change,2016-2021 Total_Pop % Change,2016-2021 Total_Pop % Change Rank,2016-2021 Total_Pop % Change Percentile
0,"Hays County, Texas",48887.0,26.327779,10.0,99.713467
1,"Comal County, Texas",32023.0,25.776358,12.0,99.649793
2,"Kaufman County, Texas",28315.0,25.319682,13.0,99.617956
3,"Osceola County, Florida",68369.0,21.915810,19.0,99.426934
4,"St. Johns County, Florida",47362.0,21.689671,20.0,99.395097
...,...,...,...,...,...
586,"Baltimore city, Maryland",-28789.0,-4.635910,2624.0,16.491563
587,"Caddo Parish, Louisiana",-13350.0,-5.274074,2709.0,13.785419
588,"Wayne County, North Carolina",-6755.0,-5.428014,2724.0,13.307864
589,"Hinds County, Mississippi",-13824.0,-5.651769,2746.0,12.607450


### Sorting counties with at least *one million* residents in 2016 by their 2016-2021 total population growth rates:

Combining `concat()` with `head()` and `tail()` will allow us to display the counties with the highest and lowest growth rates within the same DataFrame.

In [29]:
df_1m_pop_pct_changes = df_growth_data_by_year_pivot.query(
    f"Total_Pop_{range_start_year} >= 1000000").sort_values(
    total_pop_sort_column).dropna(
    subset = total_pop_sort_column).copy().reset_index()
pd.concat([df_1m_pop_pct_changes.head(), df_1m_pop_pct_changes.tail()])[
total_pop_display_cols]

,NAME,2016-2021 Total_Pop Change,2016-2021 Total_Pop % Change,2016-2021 Total_Pop % Change Rank,2016-2021 Total_Pop % Change Percentile
0,"Orange County, Florida",153894.0,12.252170,94.0,97.039160
1,"Travis County, Texas",119619.0,10.418176,140.0,95.574658
2,"Hillsborough County, Florida",121300.0,9.168147,180.0,94.301178
3,"Mecklenburg County, North Carolina",89210.0,8.817186,196.0,93.791786
4,"Clark County, Nevada",160994.0,7.776913,263.0,91.658707
38,"Miami-Dade County, Florida",25695.0,0.964376,1328.0,57.752308
39,"Cook County, Illinois",37823.0,0.723529,1394.0,55.651067
40,"Cuyahoga County, Ohio",4957.0,0.393816,1468.0,53.295129
41,"St. Louis County, Missouri",1422.0,0.142120,1543.0,50.907354
42,"Los Angeles County, California",-37520.0,-0.373068,1687.0,46.322827


## Repeating these steps in order to compare 25-to-29-year-old population growth rates:

In [30]:
df_growth_data_by_year_pivot.query(
    f"Total_Pop_{range_start_year} >= 100000").sort_values(
    young_pop_sort_column).dropna(subset = young_pop_sort_column)[
young_pop_display_cols].reset_index(drop=True)

,NAME,2016-2021 Total_Pop_25_to_29 Change,2016-2021 Total_Pop_25_to_29 % Change,2016-2021 Total_Pop_25_to_29 % Change Rank,2016-2021 Total_Pop_25_to_29 % Change Percentile
0,"Douglas County, Colorado",4368.0,31.155492,148.0,95.319962
1,"Comal County, Texas",2035.0,31.116208,150.0,95.256288
2,"Kaufman County, Texas",2062.0,30.602553,152.0,95.192614
3,"Utah County, Utah",12348.0,30.090652,161.0,94.906081
4,"Lake County, Florida",4138.0,26.023521,205.0,93.505253
...,...,...,...,...,...
586,"Arlington County, Virginia",-3287.0,-10.112602,2728.0,13.180516
587,"Lafayette Parish, Louisiana",-2209.0,-10.565334,2746.0,12.607450
588,"St. Lawrence County, New York",-745.0,-10.792409,2755.0,12.320917
589,"Caddo Parish, Louisiana",-2048.0,-11.088852,2765.0,12.002547


In [31]:
df_1m_pop_pct_changes = df_growth_data_by_year_pivot.query(
    f"Total_Pop_{range_start_year} >= 1000000").sort_values(
    young_pop_sort_column).dropna(
    subset = young_pop_sort_column).copy().reset_index()
pd.concat([df_1m_pop_pct_changes.head(), df_1m_pop_pct_changes.tail()])[
young_pop_display_cols]

,NAME,2016-2021 Total_Pop_25_to_29 Change,2016-2021 Total_Pop_25_to_29 % Change,2016-2021 Total_Pop_25_to_29 % Change Rank,2016-2021 Total_Pop_25_to_29 % Change Percentile
0,"Pima County, Arizona",9395.0,15.212850,490.0,84.431710
1,"Wayne County, Michigan",16073.0,13.496289,590.0,81.248010
2,"King County, Washington",22087.0,12.478743,643.0,79.560649
3,"Oakland County, Michigan",9476.0,12.455310,646.0,79.465138
4,"Mecklenburg County, North Carolina",10364.0,12.322545,653.0,79.242280
38,"Queens County, New York",-6216.0,-3.247598,2328.0,25.915314
39,"Allegheny County, Pennsylvania",-4548.0,-4.661699,2436.0,22.476918
40,"Miami-Dade County, Florida",-11062.0,-5.717180,2504.0,20.312003
41,"Montgomery County, Maryland",-4043.0,-6.120379,2521.0,19.770774
42,"New York County, New York",-14977.0,-7.548168,2608.0,17.000955


Although it's interesting to see which counties had the very highest and lowest growth rates, these tables provide only a limited view of overall population trends. The best way to evaluate such trends is likely via a choropleth map that will allow viewers to easily compare counties and identify regional trends. (This map will be much more intuitive to interpret than a table with thousands of rows.) We'll create these maps within the Mapping section of Python for Nonprofits.

In order to allow our mapping code to access this data, we'll first save it as a .csv file:

In [32]:
df_growth_data_by_year_pivot.to_csv(
f'Datasets/grad_destinations_acs_county_data.csv', 
    index = False)

## Retrieving state-level data

Although county-level population growth provides a fascinating look at within-state trends, the broader view that state-level data provides will also prove useful. Therefore, we'll now repurpose the functions, variable, and code created earlier to create a state-level equivalent of df_growth_data_by_year. 

In [33]:
census_data_by_year_df_list = []
for year in years_to_retrieve:
    df_data = retrieve_census_data(
        survey = 'acs5', year = year, 
        region = 'state',
        variable_list = grad_destinations_variable_list, 
        rename_data_fields = True, field_vars_dict = grad_destinations_alias_dict,
        key = key)
    census_data_by_year_df_list.append(df_data)
df_state_growth_data_by_year = pd.concat(df for df in census_data_by_year_df_list)
df_state_growth_data_by_year.query("state != '72'", inplace = True)
df_state_growth_data_by_year

,Year,NAME,state,SEX BY AGE_Estimate!!Total: (B01001_001E),SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E),SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)
1,2009,Alabama,01,4633360,155174,157476
2,2009,Alaska,02,683142,28543,24663
3,2009,Arizona,04,6324865,251627,228420
4,2009,Arkansas,05,2838143,99205,98528
5,2009,California,06,36308527,1418893,1303517
...,...,...,...,...,...,...
47,2021,Virginia,51,8582479,297545,288627
48,2021,Washington,53,7617364,297673,274857
49,2021,West Virginia,54,1801049,55187,52671
50,2021,Wisconsin,55,5871661,189562,180360


In [34]:
df_state_growth_data_by_year['Total_Pop_25_to_29'] = (df_state_growth_data_by_year[
'SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)'] + 
df_state_growth_data_by_year[
'SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'])
df_state_growth_data_by_year.rename(
    columns = {'SEX BY AGE_Estimate!!Total: (B01001_001E)':'Total_Pop'}, 
inplace = True)
df_state_growth_data_by_year.drop(
    ['SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)',
     'SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'],
     axis = 1, inplace = True)
                             

df_state_growth_data_by_year

,Year,NAME,state,Total_Pop,Total_Pop_25_to_29
1,2009,Alabama,01,4633360,312650
2,2009,Alaska,02,683142,53206
3,2009,Arizona,04,6324865,480047
4,2009,Arkansas,05,2838143,197733
5,2009,California,06,36308527,2722410
...,...,...,...,...,...
47,2021,Virginia,51,8582479,586172
48,2021,Washington,53,7617364,572530
49,2021,West Virginia,54,1801049,107858
50,2021,Wisconsin,55,5871661,369922


In [35]:
df_state_growth_data_by_year_pivot = (
    df_state_growth_data_by_year.copy().pivot(
    columns = 'Year', index = ['NAME', 'state']).reset_index())

df_state_growth_data_by_year_pivot.columns = (
    df_state_growth_data_by_year_pivot.columns.to_flat_index())

df_state_growth_data_by_year_pivot.columns = [
    column[0] + '_' + str(column[1]) if isinstance(column[1], int) 
    else column[0] for column 
    in df_state_growth_data_by_year_pivot.columns]


df_state_growth_data_by_year_pivot.head()

,NAME,state,Total_Pop_2009,Total_Pop_2011,Total_Pop_2013,Total_Pop_2015,Total_Pop_2016,Total_Pop_2021,Total_Pop_25_to_29_2009,Total_Pop_25_to_29_2011,Total_Pop_25_to_29_2013,Total_Pop_25_to_29_2015,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2021
0,Alabama,01,4633360,4747424,4799277,4830620,4841164,4997675,312650,308967,311340,314792,319177,331435
1,Alaska,02,683142,700703,720316,733375,736855,735951,53206,52613,57136,60674,61185,59218
2,Arizona,04,6324865,6337373,6479703,6641928,6728577,7079203,480047,446787,440212,447473,456680,496623
3,Arkansas,05,2838143,2895928,2933369,2958208,2968472,3006309,197733,192715,192244,192763,194179,197068
4,California,06,36308527,36969200,37659181,38421464,38654206,39455353,2722410,2737580,2779994,2866073,2918435,2992210


In [36]:
for field_var in ['Total_Pop', 'Total_Pop_25_to_29']:
    create_comparison_fields(
        df = df_state_growth_data_by_year_pivot,
        field_var = field_var,
        year_list = years_to_retrieve,
        field_year_separator='_')

Saving this dataset to a .csv file so that it too can be used as a basis for choropleth maps:

In [37]:
df_state_growth_data_by_year_pivot.to_csv(
    f'Datasets/grad_destinations_acs_state_data.csv', 
    index = False)

Creating a DataFrame that shows the 5 states with the highest and lowest total population growth rates during the latest 5 years in the dataset:

In [38]:
df_state_growth_data_by_year_pivot.sort_values(
    total_pop_sort_column, inplace = True)
df_state_growth_data_by_year_pivot.reset_index(drop=True, inplace = True)
pd.concat(
    [df_state_growth_data_by_year_pivot.head(5), 
    df_state_growth_data_by_year_pivot.tail(5)])[
total_pop_display_cols]

,NAME,2016-2021 Total_Pop Change,2016-2021 Total_Pop % Change,2016-2021 Total_Pop % Change Rank,2016-2021 Total_Pop % Change Percentile
0,Idaho,176134,10.769540,1.0,100.000000
1,Utah,282943,9.596405,2.0,98.039216
2,Nevada,220066,7.751063,3.0,96.078431
3,Washington,544218,7.694143,4.0,94.117647
4,Texas,1906146,7.071210,5.0,92.156863
46,Alaska,-904,-0.122684,47.0,9.803922
47,Illinois,-29871,-0.232429,48.0,7.843137
48,Mississippi,-22169,-0.741639,49.0,5.882353
49,Wyoming,-6388,-1.095657,50.0,3.921569
50,West Virginia,-45043,-2.439911,51.0,1.960784


Creating a similar DataFrame that displays 25-to-29-year-old population growth rates:

In [39]:
df_state_growth_data_by_year_pivot.sort_values(
    young_pop_sort_column, inplace = True)
df_state_growth_data_by_year_pivot.reset_index(drop=True, inplace = True)
pd.concat(
    [df_state_growth_data_by_year_pivot.head(5), 
    df_state_growth_data_by_year_pivot.tail(5)])[
young_pop_display_cols]

,NAME,2016-2021 Total_Pop_25_to_29 Change,2016-2021 Total_Pop_25_to_29 % Change,2016-2021 Total_Pop_25_to_29 % Change Rank,2016-2021 Total_Pop_25_to_29 % Change Percentile
0,Utah,29737,13.733051,1.0,100.000000
1,Washington,58422,11.363760,2.0,98.039216
2,Tennessee,46477,10.661207,3.0,96.078431
3,Idaho,11404,10.660534,4.0,94.117647
4,Colorado,42604,10.655262,5.0,92.156863
46,Maryland,-7907,-1.906381,47.0,9.803922
47,Hawaii,-2135,-2.059797,48.0,7.843137
48,Alaska,-1967,-3.214840,49.0,5.882353
49,Louisiana,-15558,-4.581151,50.0,3.921569
50,Wyoming,-3359,-8.256520,51.0,1.960784


# Appendix

## 1: The Census Python Library

It's worth noting that there is also a 'census' Python library (available via pypi and conda) that helps simplify the process of requesting API data. You may choose to use it for your own Census research, but I ended up not needing it for the data retrieval tasks shown above. In addition, foregoing the library allowed me to demonstrate how to retrieve data directly from an API, which you may find helpful when working with APIs that don't have a corresponding Python library. 

Here's an example of the Census library in use:

In [40]:
## Example of reading data from the Census library into a 
# Pandas DataFrame:
from census import Census
c = Census(key)
pd.DataFrame(c.acs5.get(('NAME', 'B01001_001E'),
{'for': 'county:*'}))

,NAME,B01001_001E,state,county
0,"Autauga County, Alabama",58761.0,01,001
1,"Baldwin County, Alabama",233420.0,01,003
2,"Barbour County, Alabama",24877.0,01,005
3,"Bibb County, Alabama",22251.0,01,007
4,"Blount County, Alabama",59077.0,01,009
...,...,...,...,...
3217,"Vega Baja Municipio, Puerto Rico",54182.0,72,145
3218,"Vieques Municipio, Puerto Rico",8199.0,72,147
3219,"Villalba Municipio, Puerto Rico",21984.0,72,149
3220,"Yabucoa Municipio, Puerto Rico",30313.0,72,151


## 2: The requests library

We can also use Python's *requests* library to retrieve data from the Census API, then convert it to JSON format:

In [41]:
# The following code borrows from the requests library documentation at 
# https://docs.python-requests.org/en/latest/index.html
import requests
r = requests.get(f'https://api.census.gov/data/{acs5_year}/\
acs/acs5?get=NAME,B01001_001E&for=county:*&key={key}')
# Printing the first 300 characters of this output:
print("r.text:\n",r.text[0:300],'\n')
# Printing the first 5 lines of r.json:
print("r.json:\n",r.json()[0:5],'\n')

r.text:
 [["NAME","B01001_001E","state","county"],
["Autauga County, Alabama","58239","01","001"],
["Baldwin County, Alabama","227131","01","003"],
["Barbour County, Alabama","25259","01","005"],
["Bibb County, Alabama","22412","01","007"],
["Blount County, Alabama","58884","01","009"],
["Bullock County, Ala 

r.json:
 [['NAME', 'B01001_001E', 'state', 'county'], ['Autauga County, Alabama', '58239', '01', '001'], ['Baldwin County, Alabama', '227131', '01', '003'], ['Barbour County, Alabama', '25259', '01', '005'], ['Bibb County, Alabama', '22412', '01', '007']] 



Converting our response to JSON allows it to be easily read into a Pandas DataFrame, as shown below:

In [42]:
pd.DataFrame(r.json()).head()
# Note that pd.DataFrame(r.text) would produce the following error:
# "ValueError: DataFrame constructor not properly called!"

,0,1,2,3
0,NAME,B01001_001E,state,county
1,"Autauga County, Alabama",58239,01,001
2,"Baldwin County, Alabama",227131,01,003
3,"Barbour County, Alabama",25259,01,005
4,"Bibb County, Alabama",22412,01,007


I included this approach in the appendix because you may find the requests library useful for other online data retrieval tasks. However, our use of `pd.read_json()` to import Census data rendered an explicit call to the requests library unnecessary.

In [43]:
program_end_time = time.time()
run_time = round(program_end_time - program_start_time, 3)
print(f"Finished running script in {run_time} seconds.")

Finished running script in 21.908 seconds.
